 ![1. Attention](../images/1.attention.png)

 - The figure depicts different attention mechanisms we will code in this chapter, starting
with a simplified version of self-attention before adding the trainable weights. The causal attention
mechanism adds a mask to self-attention that allows the LLM to generate one word at a time. Finally,
multi-head attention organizes the attention mechanism into multiple heads, allowing the model to
capture various aspects of the input data in parallel.

# The problem with modeling long sequences

- When translating text from one language to another, such as German to English, it’s not possible to merely translate word by word. Instead, the translation process requires contextual understanding and grammatical alignment.

 ![2. Why Attention](../images/2.lang_translation.png)

 - Before the advent of transformers, recurrent neural networks (RNNs) were the most popular encoder–decoder architecture for language translation. An RNN is a type of neural network where outputs from previous steps are fed as inputs to the current step, making them well-suited for sequential data like text.

  ![3. RNNs](../images/3.RNN.png)

- the key idea here is that the encoder part processes the entire input text into a hidden state (memory cell). The decoder then takes in this hidden state to produce the output. You can think of this hidden state as an embedding vector.

-  The big limitation of encoder–decoder RNNs is that the RNN can’t directly access earlier hidden states from the encoder during the decoding phase. Consequently, it relies solely on the current hidden state, which encapsulates all relevant information.

- This can lead to a loss of context, especially in complex sentences where dependencies might span long distances.

- One major shortcoming in this approach is that the RNN must remember the entire encoded input in a single hidden state before passing it to the decoder `Bahdanau attention mechanism`

#  Capturing data dependencies with attention mechanisms

- the original transformer architecture including a self-attention mechanism inspired by the Bahdanau attention mechanism.
- Self-attention is a mechanism in transformers used to compute more efficient input representations by allowing each position in a sequence to interact with and weigh the importance of all other positions within the same sequence. 


# Attending to different parts of the input with self-attention

- Self-attention helps a model understand how words in a sentence are related to each other. `It allows the model to look at all words at the same time and decide which ones are important` for `understanding the meaning of each word`. Because of this, `self-attention captures context more effectively` and plays a key role in models like Transformers.

# A simple self-attention mechanism without trainable weights

- In self-attention, our goal is to calculate context vectors z(i) for each element x(i) in the input sequence. A context vector can be interpreted as an enriched embedding vector.

- Context vectors play a crucial role in self-attention.

  ![5. Context vectors](../images/5.context_vectors.png)

- The main aim of context vectors is to provide a multidimensional numerical representation that captures the semantic meaning and relationships of data (such as words in a text) within a specific surrounding context (like we can predict the most relatable next word)


In [3]:
import torch
inputs = torch.tensor(
 [[0.43, 0.15, 0.89], # Your (x^1)
 [0.55, 0.87, 0.66], # journey (x^2)
 [0.57, 0.85, 0.64], # starts (x^3)
 [0.22, 0.58, 0.33], # with (x^4)
 [0.77, 0.25, 0.10], # one (x^5)
 [0.05, 0.80, 0.55]] # step (x^6)
)
inputs

tensor([[0.4300, 0.1500, 0.8900],
        [0.5500, 0.8700, 0.6600],
        [0.5700, 0.8500, 0.6400],
        [0.2200, 0.5800, 0.3300],
        [0.7700, 0.2500, 0.1000],
        [0.0500, 0.8000, 0.5500]])

-  first step of implementing self-attention is to compute the intermediate values `ω`, referred to as attention scores.

-  The overall goal is to illustrate the computation of the context vector z(2) using the second input element, x(2) as a query. This figure shows the first intermediate step, computing the attention scores  between the query x(2) and all other input elements as a dot product. 

  ![6. Context dot](../images/6.context%20dot.png)

In [28]:
# context vector for the second input
query = inputs[1]
print(f"query {query}")
attention_vec2 = torch.empty(inputs.shape[0])

for i,x_i in enumerate(inputs):
    attention_vec2[i] = torch.dot(x_i, query)

attention_vec2

query tensor([0.5500, 0.8700, 0.6600])


tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])

- `the dot product is a measure of similarity because it quantifies how closely two vectors are aligned: a higher dot product indicates a greater degree of alignment or similarity between the vectors. In the context of self-attention mechanisms, the dot product determines the extent to which each element in a sequence focuses on, or “attends to,” any other element: the
higher the dot product, the higher the similarity and attention score between two
elements.`

- we normalize each of the attention scores we computed previously. The main goal behind the normalization is to obtain attention
weights that sum up to 1. This normalization is a convention that is useful for interpretation and maintaining training stability in an LLM.

### why do we need to normalise logits?

- As you can read the scaling is to fight vanishing gradients, in fact you can interpret it as a sort of temperature that smooths the logits  𝑄𝐾⊤ making them more uniformly distributed (since a peaked distribution would cause gradient instabilities), therefore increasing the gradient magnitude.

In [30]:
attn_weights_2 = attention_vec2/attention_vec2.sum()
attn_weights_2.sum()


tensor(1.0000)

- it’s more common and advisable to use the softmax function for normalization. instead of the dividing by the sums.

In [31]:

def softmax_naive(logits):
    return torch.exp(logits)/torch.exp(logits).sum(dim=0)
# logits_tmp[3] =100
softmax_naive(attn_weights_2)
# now i am able to see difference over the distributions while normalising the logits
    

tensor([0.1630, 0.1770, 0.1765, 0.1603, 0.1570, 0.1663])

- The softmax function ensures that the attention weights are always positive. This makes the output interpretable as probabilities or relative importance, where higher weights indicate greater importance
  
- This naive softmax implementation (softmax_naive) may encounter numerical instability problems, such as overflow and underflow, when dealing with large or small input value. so use the pytorch version of softmax it handles the overflow and underflow of the large values.

In [32]:
def softmax(logits):
    return torch.softmax(logits, dim=0)

softmax(attn_weights_2)
# atleast it is handling in the softmax. for this optimisations we need the torch version of the softmax.

tensor([0.1630, 0.1770, 0.1765, 0.1603, 0.1570, 0.1663])

In [33]:
# final step of the context vector.
query = inputs[1]
context_vec_2 = torch.zeros(query.shape)
for i,x_i in enumerate(inputs):
 context_vec_2 += attn_weights_2[i]*x_i
print(context_vec_2)

tensor([0.4355, 0.6451, 0.5680])


 ![7. Context final](../images/7.context_final.png)

 # Computing attention weights for all input tokens

  ![8. Attention flow](../images/8.attention_flow.png)

- By setting dim=-1, we are instructing the softmax function to apply the normalization along the last dimension of the attn_scores tensor.

 

In [37]:
print(inputs.shape)
print(query.shape)

torch.Size([6, 3])
torch.Size([3])


In [46]:
# matrix multiplication with the inputs
attention_scores = inputs @ inputs.T

# normalise using the softmax
attention_weights = torch.softmax(attention_score, dim=-1)

# find the context vectors for inputs.
all_context_vecs =attention_weights@ inputs

attention_score, attention_weights, all_context_vecs

(tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
         [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
         [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
         [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
         [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
         [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]]),
 tensor([[0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452],
         [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581],
         [0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565],
         [0.1435, 0.2074, 0.2046, 0.1462, 0.1263, 0.1720],
         [0.1526, 0.1958, 0.1975, 0.1367, 0.1879, 0.1295],
         [0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896]]),
 tensor([[0.4421, 0.5931, 0.5790],
         [0.4419, 0.6515, 0.5683],
         [0.4431, 0.6496, 0.5671],
         [0.4304, 0.6298, 0.5510],
         [0.4671, 0.5910, 0.5266],
         [0.4177, 0.6503, 0.5645]]))

- Every thing as a summary for now.

- found the relations between the each input to other using the matrix multiplication.

- normalised those attention scores to make it as weights.

- now again converted into the token format, because at end of the day, token similarities are the main thing. that's y we done third matrix mul with the inputs and the attention_weights.